# Unitary Sequence Decomposition

Decompose a list of same-dimension unitaries into one continuous star-topology pulse sequence.

For each unitary U_k, the TAQR decomposition produces a sequence of two-level rotations plus a diagonal residual V_k. This notebook carries that diagonal residual forward as the starting virtual frame for the next unitary instead of forcing each block to close independently.

After all pulses are compiled, the exact relation is

    D_final @ U_pulses = U_N @ ... @ U_2 @ U_1

where D_final is the final residual diagonal frame.


In [18]:
import numpy as np
from numpy.linalg import norm

from U_decomp import unitary_to_G_rotations

np.set_printoptions(precision=4, suppress=True)


In [19]:
def qudit_qft(d, inverse=False):
    if d < 2:
        raise ValueError("d must be at least 2")
    omega = np.exp((-2j if inverse else 2j) * np.pi / d)
    j = np.arange(d).reshape((d, 1))
    k = np.arange(d).reshape((1, d))
    return omega ** (j * k) / np.sqrt(d)


def haar_unitary(n, rng=None):
    rng = np.random.default_rng(rng)
    X = (rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))) / np.sqrt(2)
    Q, R = np.linalg.qr(X)
    d = np.diag(R)
    D = d / np.abs(d)
    return Q * D


def haar_su(n, rng=None):
    U = haar_unitary(n, rng)
    phi = np.angle(np.linalg.det(U)) / n
    return U * np.exp(-1j * phi)

def single_qudit_phase_oracle(hidden: int, d: int) -> np.ndarray:
    """
    Phase oracle for single-qudit BV:

        O_s |x> = omega^(s*x) |x>

    where hidden = s in {0, ..., d-1}.
    """
    if d < 2:
        raise ValueError("d must be at least 2")
    if not (0 <= hidden < d):
        raise ValueError("hidden must be in {0, ..., d-1}")

    omega = np.exp(2j * np.pi / d)
    phases = np.array([omega ** (hidden * x) for x in range(d)], dtype=complex)
    return np.diag(phases)
def basis_state(index: int, d: int) -> np.ndarray:
    """
    Return |index> in dimension d.
    """
    if not (0 <= index < d):
        raise ValueError("index must be in {0, ..., d-1}")

    state = np.zeros(d, dtype=complex)
    state[index] = 1.0
    return state

In [20]:
def wrap_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi


def wrap_frame(z_frame):
    z = np.asarray(z_frame, dtype=float)
    return np.array([wrap_pi(value) for value in z], dtype=float)


def relative_frame(z_frame, reference=0):
    z = wrap_frame(z_frame)
    return wrap_frame(z - z[reference])


def virtual_z_diagonal(z_frame):
    return np.diag(np.exp(1j * np.asarray(z_frame, dtype=float)))


def ion_pulse_unitary(coupling, theta, phi, dim):
    i, j = coupling
    U = np.eye(dim, dtype=complex)
    c = np.cos(theta / 2)
    s = np.sin(theta / 2)
    U[i, i] = c
    U[j, j] = c
    U[i, j] = -1j * np.exp(1j * phi) * s
    U[j, i] = -1j * np.exp(-1j * phi) * s
    return U


def rotation_matrix_to_pulse_parameters(G, tol=1e-10):
    G = np.asarray(G, dtype=complex)
    if G.ndim != 2 or G.shape[0] != G.shape[1]:
        raise ValueError("G must be square")

    dim = G.shape[0]
    candidates = [(p, q) for p in range(dim) for q in range(p + 1, dim)
                  if abs(G[p, q]) > tol or abs(G[q, p]) > tol]
    if len(candidates) != 1:
        raise ValueError(f"Expected exactly one coupled pair, found {candidates}")

    i, j = candidates[0]
    c = G[i, i]
    s = G[i, j]
    gamma = float(wrap_pi(np.angle(c)))
    theta = float(2.0 * np.arctan2(np.clip(np.abs(s), 0.0, 1.0), np.clip(np.abs(c), 0.0, 1.0)))
    phi = float((np.angle(s) - gamma + np.pi / 2) % (2 * np.pi))

    return {
        "coupling": (i, j),
        "theta": theta,
        "fraction": float(theta / np.pi),
        "phi": phi,
        "gamma": gamma,
    }


def minimize_gamma_for_step(step):
    alternate = dict(step)
    alternate["theta"] = float(2 * np.pi - step["theta"])
    alternate["fraction"] = float(alternate["theta"] / np.pi)
    alternate["phi"] = float((step["phi"] + np.pi) % (2 * np.pi))
    alternate["gamma"] = float(wrap_pi(step["gamma"] + np.pi))
    return alternate if abs(alternate["gamma"]) < abs(step["gamma"]) else dict(step)


def diagonal_phase_vector(V, tol=1e-8):
    V = np.asarray(V, dtype=complex)
    off_diag = V - np.diag(np.diag(V))
    if norm(off_diag) > tol:
        raise ValueError(f"Residual V is not diagonal enough: {norm(off_diag)}")

    diag = np.diag(V)
    if np.max(np.abs(np.abs(diag) - 1.0)) > 1e-7:
        raise ValueError("Residual V is diagonal but not unitary on the diagonal")

    return wrap_frame(np.angle(diag))


def sequence_target_unitary(unitaries):
    total = np.eye(unitaries[0].shape[0], dtype=complex)
    for U in unitaries:
        total = U @ total
    return total


def pulse_sequence_unitary(schedule, dim):
    U = np.eye(dim, dtype=complex)
    for step in schedule:
        U = ion_pulse_unitary(step["coupling"], step["theta"], step["phi_programmed"], dim) @ U
    return U


def decompose_unitary_sequence(unitaries, center=0, tol=1e-10, minimize_gamma=True, initial_frame=None):
    if len(unitaries) == 0:
        raise ValueError("Provide at least one unitary")

    unitary_list = [np.asarray(U, dtype=complex) for U in unitaries]
    dim = unitary_list[0].shape[0]

    for idx, U in enumerate(unitary_list, start=1):
        if U.ndim != 2 or U.shape[0] != U.shape[1]:
            raise ValueError(f"Unitary {idx} is not square")
        if U.shape[0] != dim:
            raise ValueError("All unitaries must have the same dimension")

    frame = np.zeros(dim, dtype=float) if initial_frame is None else np.asarray(initial_frame, dtype=float).copy()
    if frame.shape != (dim,):
        raise ValueError("initial_frame must have shape (dim,)")

    flat_schedule = []
    block_summaries = []

    for block_idx, U in enumerate(unitary_list, start=1):
        rotation_mats, V = unitary_to_G_rotations(U, center=center, tol=tol)
        V_phase = diagonal_phase_vector(V)

        incoming_frame = frame.copy()
        frame = wrap_frame(frame + V_phase)
        compile_frame_start = frame.copy()

        target_steps = [rotation_matrix_to_pulse_parameters(G.conj().T, tol=tol) for G in rotation_mats[::-1]]
        if minimize_gamma:
            target_steps = [minimize_gamma_for_step(step) for step in target_steps]

        for step_idx, step in enumerate(target_steps, start=1):
            i, j = step["coupling"]
            phi_programmed = float((step["phi"] - (frame[i] - frame[j])) % (2 * np.pi))

            flat_schedule.append({
                "block": block_idx,
                "step_in_block": step_idx,
                "coupling": step["coupling"],
                "theta": step["theta"],
                "fraction": step["fraction"],
                "phi_programmed": phi_programmed,
                "phi_physical": step["phi"],
                "gamma": step["gamma"],
                "frame_before": frame.copy(),
            })

            frame[i] += step["gamma"]
            frame[j] -= step["gamma"]
            frame = wrap_frame(frame)
            flat_schedule[-1]["frame_after"] = frame.copy()

        block_summaries.append({
            "block": block_idx,
            "incoming_frame": incoming_frame,
            "residual_from_V": V_phase,
            "compile_frame_start": compile_frame_start,
            "outgoing_frame": frame.copy(),
            "pulse_count": len(target_steps),
        })

    return flat_schedule, frame, block_summaries


In [31]:
# Replace this list with the exact unitary sequence you want to run.
# All entries must be numpy arrays of the same dimension.
# Example: unitaries = [qudit_qft(8), qudit_qft(8, inverse=True)]

unitaries = [
    qudit_qft(8),single_qudit_phase_oracle(4, 8), qudit_qft(8, inverse=True)
]

center = 0
tol = 1e-10
minimize_gamma = True
initial_frame = None  # Or set a numpy array if you want to continue from an existing residual frame.


In [32]:
flat_schedule, final_frame, block_summaries = decompose_unitary_sequence(
    unitaries,
    center=center,
    tol=tol,
    minimize_gamma=minimize_gamma,
    initial_frame=initial_frame,
)

couplings = [step["coupling"] for step in flat_schedule]
thetas = [step["theta"] for step in flat_schedule]
fractions = [step["fraction"] for step in flat_schedule]
phases = [step["phi_programmed"] for step in flat_schedule]
gammas = [step["gamma"] for step in flat_schedule]
blocks = [step["block"] for step in flat_schedule]
steps_in_block = [step["step_in_block"] for step in flat_schedule]

print(f"number of input unitaries: {len(unitaries)}")
print(f"total pulse count: {len(flat_schedule)}")
print("final residual frame (wrapped):")
print(wrap_frame(final_frame))
print("final residual frame relative to level 0:")
print(relative_frame(final_frame, reference=0))

print("\nPer-block summary:")
for summary in block_summaries:
    print(f"block {summary['block']:2d}: pulses={summary['pulse_count']}")
    print("  residual_from_V          =", wrap_frame(summary['residual_from_V']))
    print("  compile_frame_start      =", wrap_frame(summary['compile_frame_start']))
    print("  outgoing_frame           =", wrap_frame(summary['outgoing_frame']))

print("\nFull flat schedule:")
for step in flat_schedule:
    print(
        f"block {step['block']:2d}, step {step['step_in_block']:2d}: "
        f"coupling={step['coupling']}, theta={step['theta']:.6f} rad, "
        f"fraction={step['fraction']:.6f}, phi_programmed={step['phi_programmed']:.6f} rad, "
        f"gamma={step['gamma']:+.6f} rad"
    )


number of input unitaries: 3
total pulse count: 56
final residual frame (wrapped):
[-0.     -3.1416  0.     -3.1416  0.     -3.1416  0.     -3.1416]
final residual frame relative to level 0:
[ 0.     -3.1416  0.     -3.1416  0.     -3.1416  0.     -3.1416]

Per-block summary:
block  1: pulses=28
  residual_from_V          = [1.5708 0.     0.     0.     0.     0.     0.     0.    ]
  compile_frame_start      = [1.5708 0.     0.     0.     0.     0.     0.     0.    ]
  outgoing_frame           = [-3.0335 -2.3717 -0.3408  0.4013 -0.4731  0.4115 -0.0915  0.7854]
block  2: pulses=0
  residual_from_V          = [ 0.     -3.1416  0.     -3.1416  0.     -3.1416  0.     -3.1416]
  compile_frame_start      = [-3.0335  0.7698 -0.3408 -2.7403 -0.4731 -2.7301 -0.0915 -2.3562]
  outgoing_frame           = [-3.0335  0.7698 -0.3408 -2.7403 -0.4731 -2.7301 -0.0915 -2.3562]
block  3: pulses=28
  residual_from_V          = [-1.5708  0.      0.      0.      0.      0.      0.      0.    ]
  compile_frame

In [33]:
print("couplings =", couplings)
print("fractions =", fractions)
print("thetas =", thetas)
print("phases =", phases)
print("gammas =", gammas)
print("blocks =", blocks)
print("steps_in_block =", steps_in_block)
print("final_frame =", wrap_frame(final_frame))
print("final_frame_relative =", relative_frame(final_frame, reference=0))


couplings = [(0, 1), (0, 2), (0, 1), (0, 3), (0, 2), (0, 1), (0, 4), (0, 3), (0, 2), (0, 1), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 6), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 7), (0, 6), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 1), (0, 2), (0, 1), (0, 3), (0, 2), (0, 1), (0, 4), (0, 3), (0, 2), (0, 1), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 6), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 7), (0, 6), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1)]
fractions = [1.632145581186304, 1.587198145049057, 0.8311807099003125, 0.6089050782275527, 0.2867626030493737, 0.674308567431865, 0.6736013739105923, 0.3138359498434048, 0.3354391057710658, 0.5649491093223904, 0.7032106395291597, 0.32151430649137375, 0.3597616411955304, 0.4254566937704658, 0.38658373907280985, 1.277500198538068, 0.32807406686498514, 0.37571627661241297, 0.4094400916625794, 0.40127699726373894, 1.7016613817989037, 0.7699465438373836, 0.24675171442885052, 0.2677204728012306, 0.29516723530086714, 0.3333333333333338, 0.3

In [34]:
dim = unitaries[0].shape[0]
U_pulses = pulse_sequence_unitary(flat_schedule, dim)
U_target = sequence_target_unitary(unitaries)
U_exact = virtual_z_diagonal(final_frame) @ U_pulses

print("pulse-only error vs full target:", norm(U_pulses - U_target))
print("pulse-plus-final-frame error vs full target:", norm(U_exact - U_target))


pulse-only error vs full target: 4.000000000000009
pulse-plus-final-frame error vs full target: 1.4956145115892404e-14


In [35]:
U_pulses @ basis_state(0, 8)

array([ 0.-0.j, -0.-0.j,  0.+0.j, -0.-0.j,  1.-0.j,  0.+0.j,  0.+0.j,
        0.+0.j])